# Embed the reviews and use a regression model to predict the final satisfaction

In [2]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import pandas as pd
import os
import re
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
import random
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sentence_transformers import SentenceTransformer, util
import torch


import warnings

warnings.filterwarnings("ignore")

/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-02-04 15:41:39.065789: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-04 15:41:39.092588: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To e

# TRAINING

## DOWNLOAD TRAIN DATASET TO ADD FEATURES

In [3]:
df = pd.read_csv('../data/cleaned_data/cleaned_Study_1_reviews.csv')

## X and y separation

Maybe later we can add attribute presence in the features, but for the moment X is only the embedding representation of reviews

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", cache_folder='../embedding_models').to(device)

In [ ]:
df_X = model.encode(
    df["finalReview"].fillna("").tolist(), 
    # convert_to_tensor=True
    convert_to_tensor=False
)
df_y = df["Satisfaction_final"]

print(len(df_X))
print(len(df_y))

2602
2602


# TESTING MODELS 

## RF

### GRID SEARCH

In [28]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_predict

X = df_X
y = df_y.values

rf_clf = RandomForestRegressor(random_state=42)

param_grid = {
    "n_estimators": [100, 200],  # Nombre d'arbres dans la forêt
    "max_depth": [10, 20],  # Profondeur maximale des arbres
    "min_samples_split": [2, 5],  # Nombre minimal d'échantillons nécessaires pour diviser un nœud
    "min_samples_leaf": [2, 4],  # Nombre minimal d'échantillons nécessaires dans un feuille
    "max_features": ["sqrt", "log2"],  # Optionnel: ajouter des options pour max_features
}

# Pour la régression, utilisez "neg_mean_absolute_error" (pas "mae")
grid_search_rf = GridSearchCV(
    estimator=rf_clf,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",  # ou "neg_mean_squared_error", "r2"
    n_jobs=-1,
    verbose=2,
)

grid_search_rf.fit(X, y)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=100; total time=   2.3s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=100; total time=   2.3s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=100; total time=   2.3s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   2.4s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   2.4s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   2.4s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=100; total time=   2.4s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=10

GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [2, 4],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='neg_mean_absolute_error', verbose=2)

In [ ]:
print("Meilleurs paramètres trouvés :", grid_search_rf.best_params_)
print("Meilleur score MAE (cross-validation) :", -grid_search_rf.best_score_)  # Négatif -> positif

best_rf_clf = grid_search_rf.best_estimator_

rf_cv_mae = cross_val_score(best_rf_clf, X, y, cv=10, scoring="neg_mean_absolute_error")

# Obtenir les prédictions avec cross-validation pour calculer le MAE avec arrondi
y_pred_cv = cross_val_predict(best_rf_clf, X, y, cv=10)

# Arrondir les prédictions au 0.5 le plus proche
y_pred_rounded = np.round(y_pred_cv * 2) / 2

# Calculer le MAE avec les prédictions arrondies
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print("\n=== Résultats avec Grid Search ===")
print("Random Forest - MAE moyenne (cross-validation) :", -np.mean(rf_cv_mae))
print("Random Forest - Ecart-type MAE :", np.std(rf_cv_mae))
print("\n=== Avec arrondi à 0.5 ===")
print("Random Forest - MAE avec prédictions arrondies à 0.5 :", mae_rounded)

# Optionnel: voir la distribution des prédictions arrondies
print("\nDistribution des prédictions arrondies:")
unique, counts = np.unique(y_pred_rounded, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  Note {val}: {count} prédictions ({count/len(y_pred_rounded)*100:.1f}%)")

Meilleurs paramètres trouvés : {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Meilleur score MAE (cross-validation) : 0.9899196195707957

=== Résultats avec Grid Search ===
Random Forest - MAE moyenne (cross-validation) : 0.9813709943382772
Random Forest - Ecart-type MAE : 0.055776392255955626
Random Forest - RMSE moyenne (cross-validation) : 1.3235683084522107
Random Forest - R² moyen (cross-validation) : 0.6370516971397153
Random Forest - Ecart-type R² : 0.027466984997416597

=== Avec arrondi à 0.5 ===
Random Forest - MAE avec prédictions arrondies à 0.5 : 0.9754035357417371

Distribution des prédictions arrondies:
  Note 1.5: 1 prédictions (0.0%)
  Note 2.0: 6 prédictions (0.2%)
  Note 3.0: 4 prédictions (0.2%)
  Note 3.5: 78 prédictions (3.0%)
  Note 4.0: 339 prédictions (13.0%)
  Note 4.5: 430 prédictions (16.5%)
  Note 5.0: 262 prédictions (10.1%)
  Note 5.5: 194 prédictions (7.5%)
  Note 6.0: 211 prédictions (8.1%)
 

Extended parameter search (select randomly parameters in the grid)

In [ ]:
# # Expanded parameter grid - Strategic exploration
# param_grid = {
#     "n_estimators": [200, 300, 500],  # 200 était optimal, essayer plus haut
#     "max_depth": [20, 30, 40, None],  # None = profondeur illimitée
#     "min_samples_split": [2, 5, 10],  # 5 était optimal
#     "min_samples_leaf": [1, 2, 4],  # Essayer 1 (plus flexible)
#     "max_features": ["sqrt", "log2", 0.5],  # Ajouter une option numérique
#     "min_impurity_decrease": [0.0, 0.001, 0.01],  # Nouveau: contrôle la division des nœuds
# }

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV
# from scipy.stats import randint, uniform

# # Distributions pour RandomizedSearchCV (plus efficace)
# param_distributions = {
#     "n_estimators": randint(200, 600),  # Échantillonne entre 200-600
#     "max_depth": [20, 30, 40, 50, None],
#     "min_samples_split": randint(2, 20),
#     "min_samples_leaf": randint(1, 10),
#     "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
#     "min_impurity_decrease": uniform(0, 0.01),  # Entre 0 et 0.01
#     "max_samples": uniform(0.7, 0.3),  # Bootstrap sample size (0.7 à 1.0)
# }

# random_search = RandomizedSearchCV(
#     estimator=rf_clf,
#     param_distributions=param_distributions,
#     n_iter=100,  # Teste 100 combinaisons aléatoires
#     cv=5,
#     scoring="neg_mean_absolute_error",
#     n_jobs=-1,
#     verbose=2,
#     random_state=42
# )

# random_search.fit(X, y)

# print("Meilleurs paramètres trouvés :", random_search.best_params_)
# print("Meilleur score MAE :", -random_search.best_score_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END max_depth=None, max_features=log2, max_samples=0.9165996316800473, min_impurity_decrease=0.009385527090157502, min_samples_leaf=2, min_samples_split=2, n_estimators=513; total time=   4.1s
[CV] END max_depth=None, max_features=log2, max_samples=0.9165996316800473, min_impurity_decrease=0.009385527090157502, min_samples_leaf=2, min_samples_split=2, n_estimators=513; total time=   4.3s
[CV] END max_depth=None, max_features=log2, max_samples=0.9165996316800473, min_impurity_decrease=0.009385527090157502, min_samples_leaf=2, min_samples_split=2, n_estimators=513; total time=   4.4s
[CV] END max_depth=None, max_features=log2, max_samples=0.9165996316800473, min_impurity_decrease=0.009385527090157502, min_samples_leaf=2, min_samples_split=2, n_estimators=513; total time=   4.4s
[CV] END max_depth=None, max_features=log2, max_samples=0.9165996316800473, min_impurity_decrease=0.009385527090157502, min_samples_leaf=2, min_s

In [ ]:
# # Vérifier si le modèle actuel overfitte
# from sklearn.model_selection import learning_curve

# train_sizes, train_scores, val_scores = learning_curve(
#     best_rf_clf, X, y, 
#     cv=5, 
#     scoring="neg_mean_absolute_error",
#     train_sizes=np.linspace(0.1, 1.0, 10),
#     n_jobs=-1
# )

# train_mae = -train_scores.mean(axis=1)
# val_mae = -val_scores.mean(axis=1)

# print("MAE sur train (dernier point):", train_mae[-1])
# print("MAE sur validation (dernier point):", val_mae[-1])
# print("Écart (overfitting si > 0.3):", val_mae[-1] - train_mae[-1])

MAE sur train (dernier point): 0.4301159249773091
MAE sur validation (dernier point): 0.9908928584546952
Écart (overfitting si > 0.3): 0.5607769334773861


## GRADIENT BOOSTING

### GRID SEARCH

In [45]:
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error

X = df_X
y = df_y.values

gb_clf = GradientBoostingRegressor(random_state=50)
gb_clf = XGBRegressor(random_state=50)

# Paramètres adaptés pour la régression
param_grid = {
    "n_estimators": [100, 200, 300],  # Plus d'arbres souvent mieux pour GB
    "learning_rate": [0.01, 0.05, 0.1],  # Taux d'apprentissage
    "max_depth": [3, 4],  # Profondeur des arbres (GB fonctionne bien avec des arbres peu profonds)
    # "min_samples_split": [2, 5],
    # "min_samples_leaf": [1, 2],
    "subsample": [0.8, 1.0],  # Fraction d'échantillons pour chaque arbre
    # "max_features": ["sqrt", "log2"],  # Optionnel mais peut améliorer
}

grid_search_xgb = GridSearchCV(
    estimator=gb_clf,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2,
)

grid_search_xgb.fit(X, y)


Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   5.8s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   6.4s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=   8.9s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   8.9s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   9.2s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=   9.2s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=   9.4s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=   9.8s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=  10.0s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=  10.1

GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, gamma=None,
                                    grow_policy=None, importance_type=None,
                                    interaction_constraints=None,
                                    learning_rate=None, m...
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None,
                                    random_state=50, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [3, 4], 'n_estimators': [100, 200, 300],
                         'subsample': [0.8, 1.0]},
             scoring='neg_mean_absolute_error', verbose=2)

In [46]:
print("Meilleurs paramètres trouvés :", grid_search_xgb.best_params_)
print("Meilleur score MAE (cross-validation) :", -grid_search_xgb.best_score_)

best_xgb_clf = grid_search_xgb.best_estimator_

# Métriques de régression
xgb_cv_mae = cross_val_score(best_xgb_clf, X, y, cv=10, scoring="neg_mean_absolute_error")
# gb_cv_mse = cross_val_score(best_gb_clf, X, y, cv=10, scoring="neg_mean_squared_error")
# gb_cv_r2 = cross_val_score(best_gb_clf, X, y, cv=10, scoring="r2")

# Prédictions avec arrondi à 0.5
y_pred_cv = cross_val_predict(best_xgb_clf, X, y, cv=10)
y_pred_rounded = np.round(y_pred_cv * 2) / 2
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print("\n=== Résultats Gradient Boosting avec Grid Search ===")
print("Gradient Boosting - MAE moyenne (cross-validation) :", -np.mean(xgb_cv_mae))
print("Gradient Boosting - Ecart-type MAE :", np.std(xgb_cv_mae))
# print("Gradient Boosting - RMSE moyenne (cross-validation) :", np.sqrt(-np.mean(gb_cv_mse)))
# print("Gradient Boosting - R² moyen (cross-validation) :", np.mean(gb_cv_r2))
# print("Gradient Boosting - Ecart-type R² :", np.std(gb_cv_r2))
print("\n=== Avec arrondi à 0.5 ===")
print("Gradient Boosting - MAE avec prédictions arrondies à 0.5 :", mae_rounded)

# Distribution des prédictions arrondies
print("\nDistribution des prédictions arrondies:")
unique, counts = np.unique(y_pred_rounded, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  Note {val}: {count} prédictions ({count/len(y_pred_rounded)*100:.1f}%)")

Meilleurs paramètres trouvés : {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.8}
Meilleur score MAE (cross-validation) : 0.911747824865387

=== Résultats Gradient Boosting avec Grid Search ===
Gradient Boosting - MAE moyenne (cross-validation) : 0.9000014625852909
Gradient Boosting - Ecart-type MAE : 0.04070524921032721

=== Avec arrondi à 0.5 ===
Gradient Boosting - MAE avec prédictions arrondies à 0.5 : 0.893543428132206

Distribution des prédictions arrondies:
  Note 0.5: 1 prédictions (0.0%)
  Note 1.0: 9 prédictions (0.3%)
  Note 1.5: 3 prédictions (0.1%)
  Note 2.0: 14 prédictions (0.5%)
  Note 2.5: 46 prédictions (1.8%)
  Note 3.0: 98 prédictions (3.8%)
  Note 3.5: 175 prédictions (6.7%)
  Note 4.0: 258 prédictions (9.9%)
  Note 4.5: 275 prédictions (10.6%)
  Note 5.0: 236 prédictions (9.1%)
  Note 5.5: 158 prédictions (6.1%)
  Note 6.0: 168 prédictions (6.5%)
  Note 6.5: 213 prédictions (8.2%)
  Note 7.0: 222 prédictions (8.5%)
  Note 7.5: 249 prédi

In [47]:
X = df_X
y = df_y.values

xgb_model = GradientBoostingRegressor(random_state=50, learning_rate=0.05, max_depth=5, n_estimators=300, subsample=0.8)

xgb_model.fit(X, y)

y_pred = xgb_model.predict(X)
y_pred_rounded = np.round(y_pred_cv * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.22894006449218748
mae with rounded predictions: 0.893543428132206


## LINEAR REGRESSION

### GRID SEARCH

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression

# Test plusieurs modèles de régression linéaire
models = {
    "Linear Regression": (LinearRegression(), {}),  # Pas de paramètres à tuner
    "Ridge": (Ridge(random_state=42), {
        "alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
        "solver": ["auto", "svd", "cholesky", "lsqr", "saga"],
    }),
    "Lasso": (Lasso(random_state=42, max_iter=5000), {
        "alpha": [0.001, 0.01, 0.1, 1, 10, 100],
    }),
    "ElasticNet": (ElasticNet(random_state=42, max_iter=5000), {
        "alpha": [0.001, 0.01, 0.1, 1, 10],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],  # Mix entre L1 et L2
    }),
}

results = {}

for name, (model, param_grid) in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print('='*50)
    
    if param_grid:  # Si des paramètres à tuner
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=5,
            scoring="neg_mean_absolute_error",
            verbose=0,
            n_jobs=-1,
        )
        grid_search.fit(X, y)
        best_model = grid_search.best_estimator_
        print(f"Meilleurs paramètres: {grid_search.best_params_}")
    else:  # LinearRegression simple
        best_model = model
        best_model.fit(X, y)
    
    # Évaluation
    cv_mae = cross_val_score(best_model, X, y, cv=10, scoring="neg_mean_absolute_error")
    
    # Prédictions arrondies
    y_pred_cv = cross_val_predict(best_model, X, y, cv=10)
    y_pred_rounded = np.round(y_pred_cv * 2) / 2
    mae_rounded = mean_absolute_error(y, y_pred_rounded)
    
    results[name] = {
        "MAE": -np.mean(cv_mae),
        "MAE_rounded": mae_rounded,
    }
    
    print(f"MAE: {-np.mean(cv_mae):.4f}")
    print(f"MAE (arrondi 0.5): {mae_rounded:.4f}")

# Résumé comparatif
print("\n" + "="*60)
print("RÉSUMÉ COMPARATIF")
print("="*60)
for name, metrics in results.items():
    print(f"{name:20s} | MAE: {metrics['MAE']:.4f} | MAE_rounded: {metrics['MAE_rounded']:.4f}")


Training Linear Regression...
MAE: 1.0555
R²: 0.5808
MAE (arrondi 0.5): 1.0505

Training Ridge...


/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Meilleurs paramètres: {'alpha': 1, 'solver': 'auto'}
MAE: 0.9072
R²: 0.6997
MAE (arrondi 0.5): 0.9012

Training Lasso...
Meilleurs paramètres: {'alpha': 0.001}
MAE: 0.9504
R²: 0.6777
MAE (arrondi 0.5): 0.9414

Training ElasticNet...
Meilleurs paramètres: {'alpha': 0.001, 'l1_ratio': 0.1}
MAE: 0.9285
R²: 0.6927
MAE (arrondi 0.5): 0.9247

RÉSUMÉ COMPARATIF
Linear Regression    | MAE: 1.0555 | R²: 0.5808 | MAE_rounded: 1.0505
Ridge                | MAE: 0.9072 | R²: 0.6997 | MAE_rounded: 0.9012
Lasso                | MAE: 0.9504 | R²: 0.6777 | MAE_rounded: 0.9414
ElasticNet           | MAE: 0.9285 | R²: 0.6927 | MAE_rounded: 0.9247


In [43]:
X = df_X
y = df_y.values

clf = Ridge(random_state=42, alpha=1, solver="auto")

clf.fit(X, y)

y_pred = clf.predict(X)
y_pred_rounded = np.round(y_pred_cv * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.8033246697140692
mae with rounded predictions: 0.92467332820907


## XGBOOST

### GRID SEARCH

In [18]:
X = df_X.values
y = df_y.values

xgb_clf = XGBClassifier(
    random_state=42, n_estimators=100, use_label_encoder=False, eval_metric="logloss"
)

param_grid = {
    "n_estimators": [100, 200],  # Nombre d'estimateurs (arbres)
    "learning_rate": [0.1, 0.2],  # Taux d'apprentissage
    "max_depth": [3, 4],  # Profondeur maximale des arbres
    "min_child_weight": [1],  # Poids minimal d'un enfant
    "subsample": [
        1.0,
    ],  # Fraction d'échantillons utilisée pour entraîner chaque arbre
    "colsample_bytree": [
        0.8,
    ],  # Fraction des colonnes utilisées pour chaque arbre
}

grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
)

grid_search.fit(X, y)

print("Meilleurs paramètres trouvés :", grid_search.best_params_)
print("Meilleure précision moyenne (cross-validation) :", grid_search.best_score_)

best_xgb_clf = grid_search.best_estimator_

xgb_cv_scores = cross_val_score(best_xgb_clf, X, y, cv=10, scoring="accuracy")
print(
    "XGBoost - Accuracy moyenne (cross-validation) avec Grid Search :",
    np.mean(xgb_cv_scores),
)
print(
    "XGBoost - Ecart-type de l'Accuracy (cross-validation) avec Grid Search :",
    np.std(xgb_cv_scores),
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Meilleurs paramètres trouvés : {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 1.0}
Meilleure précision moyenne (cross-validation) : 0.5975913239510604
XGBoost - Accuracy moyenne (cross-validation) avec Grid Search : 0.5583980518625774
XGBoost - Ecart-type de l'Accuracy (cross-validation) avec Grid Search : 0.09414449374914556
